In [1]:
from huggingface_hub import login
login()

In [2]:
from smolagents import CodeAgent, Tool, InferenceClientModel
import requests
import json
from datetime import datetime
from typing import Optional
import os
import random
import time

In [3]:
#配置 Langfuse
LANGFUSE_HOST = "https://cloud.langfuse.com"
LANGFUSE_PUBLIC_KEY = "pk-lf-cd42c76b-7a70-4de8-8917-b0078fc7b548"
LANGFUSE_SECRET_KEY = "sk-lf-1192c0c2-e43c-4873-98b3-096e10f544be"

In [4]:
from langfuse import Langfuse

try:
    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
    )
    print("✅ Langfuse 初始化成功！")
except Exception as e:
    print(f"⚠️ Langfuse 初始化警告: {e}")
    # 创建一个虚拟的 Langfuse 对象，避免程序崩溃
    class DummyLangfuse:
        def trace(self, **kwargs):
            return DummyTrace()
        def flush(self):
            pass
    
    class DummyTrace:
        def update(self, **kwargs):
            pass
    
    langfuse = DummyLangfuse()
    print("⚠️ 使用虚拟模式运行（追踪功能已禁用）")

✅ Langfuse 初始化成功！


Weather

In [5]:
from smolagents import tool

@tool
def get_current_weather(city: str) -> str:
    """
    查询指定城市的实时天气信息。
    Args:
        city: 城市名称（英文或中文拼音，如 'Beijing', 'Shanghai'）
    """
    try:
        url = f"https://wttr.in/{city}?format=%C:+%t,+湿度:%h,+风速:%w&lang=zh"
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            weather_data = response.text.strip()
            return f"📍 {city} 的天气：{weather_data}"
        else:
            # 备用 API
            url = f"https://wttr.in/{city}?format=3"
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                return f"📍 {city} 的天气：{response.text.strip()}"
            else:
                return f"❌ 无法获取 {city} 的天气信息"
    except Exception as e:
        return f"❌ 查询天气时出错：{str(e)}"

English sentence

In [6]:
@tool
def get_meaningful_english_sentence(category: str = "inspirational") -> str:
    """
    获取富有内涵或接地气的英文句子，附带中文解释。
    50%概率从网络搜索，50%概率使用AI生成。
    Args:
        category: 句子类型，可选 'inspirational'(励志), 'wisdom'(智慧), 'slang'(俚语), 'idiom'(习语)
    """
    
    # 随机决定使用搜索还是AI生成
    use_search = random.choice([True, False])
    
    if use_search:
        print("🔍 正在从网络搜索真实英文句子...")
        result = search_from_network(category)
        if result:
            return f"🌐 [网络搜索] {result}"
        else:
            # 如果搜索失败，降级到AI生成
            print("⚠️ 网络搜索失败，切换到AI生成...")
            result = generate_with_ai(category)
            return f"🤖 [AI生成] {result}" if result else get_fallback_sentence(category)
    else:
        print("🤖 正在使用AI生成英文句子...")
        result = generate_with_ai(category)
        if result:
            return f"🤖 [AI生成] {result}"
        else:
            # 如果AI生成失败，降级到网络搜索
            print("⚠️ AI生成失败，切换到网络搜索...")
            result = search_from_network(category)
            return f"🌐 [网络搜索] {result}" if result else get_fallback_sentence(category)

def search_from_network(category: str) -> str:
    """从网络搜索真实的英文句子"""
    import requests
    import re
    
    # 根据类别设置搜索关键词
    search_keywords = {
        "inspirational": "inspirational quotes",
        "wisdom": "wise sayings proverbs",
        "slang": "english slang expressions",
        "idiom": "english idioms"
    }
    
    keyword = search_keywords.get(category, "english quotes")
    
    try:
        # 使用多个免费的 Quotes API
        
        # API 1: Quotable API (免费，无需key)
        if category in ["inspirational", "wisdom"]:
            api_url = f"https://api.quotable.io/random?tags={category}"
            try:
                response = requests.get(api_url, timeout=5)
                if response.status_code == 200:
                    data = response.json()
                    quote = data.get('content', '')
                    author = data.get('author', 'Unknown')
                    
                    # 简单的翻译（使用免费翻译API或本地处理）
                    translation = translate_sentence(quote)
                    
                    return f"📖 {quote} - {author}\n\n💡 中译：{translation}"
            except:
                pass
        
        # API 2: Type.fit API (免费)
        if category == "inspirational":
            try:
                response = requests.get("https://type.fit/api/quotes", timeout=5)
                if response.status_code == 200:
                    quotes = response.json()
                    import random
                    quote_data = random.choice(quotes)
                    quote = quote_data.get('text', '')
                    author = quote_data.get('author', 'Unknown')
                    translation = translate_sentence(quote)
                    return f"📖 {quote} - {author}\n\n💡 中译：{translation}"
            except:
                pass
        
        # API 3: 针对俚语的 Urban Dictionary
        if category == "slang":
            slang_words = ["lit", "extra", "clout", "dope", "GOAT", "bet", "cap"]
            import random
            word = random.choice(slang_words)
            url = f"https://api.urbandictionary.com/v0/define?term={word}"
            try:
                response = requests.get(url, timeout=5)
                if response.status_code == 200:
                    data = response.json()
                    if data['list']:
                        definition = data['list'][0]['definition']
                        example = data['list'][0].get('example', 'No example')
                        # 清理HTML标签
                        definition = re.sub(r'\[.*?\]', '', definition)
                        example = re.sub(r'\[.*?\]', '', example)
                        return f"📖 俚语：'{word}'\n💡 定义：{definition[:150]}...\n📝 例句：{example[:150]}..."
            except:
                pass
        
        # 如果API都失败，使用备用网络搜索（模拟）
        return get_cached_sentences(category)
        
    except Exception as e:
        print(f"网络搜索错误: {e}")
        return None

def generate_with_ai(category: str) -> str:
    """使用AI模型生成英文句子"""
    try:
        # 使用 Hugging Face 模型
        model = InferenceClientModel(model_id="Qwen/Qwen2.5-7B-Instruct")
        
        # 根据类别设计不同的prompt
        prompts = {
            "inspirational": """Generate an original, inspirational English quote with Chinese translation. 
Format: 
English: [your quote]
Chinese: [Chinese translation]

Example:
English: Dream big, work hard, stay focused.
Chinese: 梦想远大，努力工作，保持专注。

Now generate a different inspirational quote:""",
            
            "wisdom": """Generate a wise English proverb with Chinese translation.
Format:
English: [proverb]
Chinese: [Chinese translation]

Generate:""",
            
            "slang": """Generate a modern, authentic English slang expression with its meaning and Chinese translation.
Format:
English: [slang expression]
Meaning: [explanation in English]
Chinese: [Chinese translation]

Generate:""",
            
            "idiom": """Generate a common English idiom with Chinese translation and example.
Format:
English: [idiom]
Chinese: [Chinese translation]
Example: [example sentence]

Generate:"""
        }
        
        prompt = prompts.get(category, prompts["inspirational"])
        
        # 调用模型生成
        response = model.generate(prompt, max_new_tokens=200, temperature=0.8)
        
        # 解析响应
        return parse_ai_response(response, category)
        
    except Exception as e:
        print(f"AI生成错误: {e}")
        return None

def parse_ai_response(response: str, category: str) -> str:
    """解析AI生成的响应"""
    lines = response.strip().split('\n')
    
    if category == "slang":
        # 俚语的特殊处理
        english = ""
        meaning = ""
        chinese = ""
        for line in lines:
            if line.startswith("English:"):
                english = line.replace("English:", "").strip()
            elif line.startswith("Meaning:"):
                meaning = line.replace("Meaning:", "").strip()
            elif line.startswith("Chinese:"):
                chinese = line.replace("Chinese:", "").strip()
        
        if english:
            result = f"📖 {english}\n"
            if meaning:
                result += f"💡 含义：{meaning}\n"
            if chinese:
                result += f"💡 中译：{chinese}"
            return result
    
    elif category in ["inspirational", "wisdom"]:
        english = ""
        chinese = ""
        for line in lines:
            if line.startswith("English:"):
                english = line.replace("English:", "").strip()
            elif line.startswith("Chinese:"):
                chinese = line.replace("Chinese:", "").strip()
        
        if english:
            return f"📖 {english}\n💡 中译：{chinese if chinese else '（翻译生成中...）'}"
    
    # 如果解析失败，直接返回原始响应
    return f"📖 {response[:300]}"

def translate_sentence(text: str) -> str:
    """简单的句子翻译（使用免费API）"""
    try:
        # 使用免费的 MyMemory API 进行翻译
        url = f"https://api.mymemory.translated.net/get?q={text}&langpair=en|zh"
        response = requests.get(url, timeout=3)
        if response.status_code == 200:
            data = response.json()
            translation = data.get('responseData', {}).get('translatedText', '')
            if translation and translation != text:
                return translation[:200]
    except:
        pass
    
    # 如果翻译失败，返回提示
    return "（可手动翻译）"

def get_cached_sentences(category: str) -> str:
    """缓存的句子库（作为备用）"""
    import random
    
    cache = {
        "inspirational": [
            ("Believe you can and you're halfway there.", "相信你能做到，你就已经成功了一半。", "Theodore Roosevelt"),
            ("The future belongs to those who believe in their dreams.", "未来属于那些相信自己梦想的人。", "Eleanor Roosevelt"),
            ("Success is not final, failure is not fatal.", "成功不是终点，失败也非末日。", "Winston Churchill")
        ],
        "wisdom": [
            ("The only true wisdom is in knowing you know nothing.", "真正的智慧是认识到自己的无知。", "Socrates"),
            ("Knowledge speaks, but wisdom listens.", "知识会说话，但智慧会倾听。", "Jimi Hendrix")
        ],
        "slang": [
            ("No cap", "说实话，不骗你", "Meaning: No lie"),
            ("That's fire", "太棒了", "Meaning: Amazing"),
            ("I'm dead", "笑死了", "Meaning: Extremely funny")
        ],
        "idiom": [
            ("Break a leg", "祝你好运", "Used to wish good luck"),
            ("Piece of cake", "小菜一碟", "Meaning: Very easy"),
            ("Hit the sack", "睡觉去", "Meaning: Go to sleep")
        ]
    }
    
    items = cache.get(category, cache["inspirational"])
    item = random.choice(items)
    
    if len(item) == 3:
        eng, chn, extra = item
        return f"📖 {eng}\n💡 中译：{chn}\n💡 备注：{extra}"
    else:
        eng, chn = item
        return f"📖 {eng}\n💡 中译：{chn}"

def get_fallback_sentence(category: str) -> str:
    """最终的备用句子（当所有方法都失败时）"""
    fallback = {
        "inspirational": "📖 Every day is a new beginning.\n💡 中译：每一天都是新的开始。",
        "wisdom": "📖 Time waits for no one.\n💡 中译：时间不等人。",
        "slang": "📖 You got this!\n💡 中译：你可以的！",
        "idiom": "📖 Practice makes perfect.\n💡 中译：熟能生巧。"
    }
    return fallback.get(category, fallback["inspirational"])

In [7]:
print("🤖 正在创建智能Agent...")

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-7B-Instruct",
    max_tokens=5000,
    temperature=0.7
)

agent = CodeAgent(
    tools=[
        get_current_weather,
        get_meaningful_english_sentence
    ],
    model=model,
    max_steps=10,
    verbosity_level=2,
    additional_authorized_imports=['requests', 'random', 'json', 're', 'time']
)

print("✅ Agent创建成功！")

🤖 正在创建智能Agent...
✅ Agent创建成功！


In [8]:
def test_agent():
    """测试Agent的各种功能"""
    
    print("\n" + "="*70)
    print("🧪 开始测试 - 英文句子获取（会随机使用搜索或AI生成）")
    print("="*70)
    
    # 测试多次以显示随机性
    categories = ["inspirational", "slang", "wisdom", "idiom"]
    
    for i in range(5):
        print(f"\n📝 测试 {i+1}/5:")
        category = random.choice(categories)
        result = agent.run(f"给我一句{category}类型的英文句子")
        print(f"结果：{result}\n")
        print("-"*50)
        time.sleep(1)  # 避免请求过快
    
    # 测试综合任务
    print("\n" + "="*70)
    print("📋 综合测试：查询天气 + 获取英文句子")
    print("="*70)
    
    result = agent.run("""
    请帮我：
    1. 查询北京的天气
    2. 给我一句励志的英文句子
    3. 根据天气给出活动建议
    """)
    print(f"\n结果：{result}")

def interactive_mode():
    """交互式模式"""
    print("\n" + "="*70)
    print("🤖 进入交互式模式")
    print("你可以问我关于天气或英文句子的问题")
    print("每次获取英文句子，我会随机选择网络搜索或AI生成")
    print("输入 'exit' 退出")
    print("="*70)
    
    while True:
        question = input("\n💬 你的问题: ")
        if question.lower() == 'exit':
            break
        
        print("\n🔄 处理中...")
        result = agent.run(question)
        print(f"\n🤖 回答:\n{result}")

In [ ]:
if __name__ == "__main__":
    print("\n🌟 欢迎使用Agent")
    
    choice = input("\n请选择模式:\n1 - 测试模式（演示随机性）\n2 - 交互式模式\n3 - 快速测试单个句子\n请输入 (1/2/3): ")
    
    if choice == "1":
        test_agent()
    elif choice == "2":
        interactive_mode()
    elif choice == "3":
        print("\n快速测试：")
        for cat in ["inspirational", "slang", "wisdom"]:
            print(f"\n测试 {cat}:")
            result = agent.run(f"给我一句{cat}类型的英文句子")
            print(result)
    else:
        test_agent()
    
    print("\n" + "="*70)
    print("🎉 程序运行完成！")


🌟 欢迎使用Agent

🧪 开始测试 - 英文句子获取（会随机使用搜索或AI生成）

📝 测试 1/5:


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ 给我一句inspirational类型的英文句子                                                                             │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-7B-Instruct ───────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: 我将使用 `get_meaningful_english_sentence` 工具来获取一句励志类型的英文句子。                             
<code>                                                                                                             
sentence = get_meaningful_english_sentence(category="inspirational")                                               
print(sentence)                                                                                                    
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sentence = get_meaningful_english_sentence(category="inspirational")                                             
  print(sentence)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "北京天气": weather_in_beijing,                                                                              
      "励志英文句子": motivational_sentence,                                                                       
      "活动建议": "建议去户外活动，享受阳光和温暖的天气。"                                                         
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'北京天气': '📍 北京 的天气：Sunny: +26Â°C, æ¹¿åº¦:45%, é£\x8eé\x80\x9f:â\x86\x9122km/h', 
'励志英文句子': '🌐 [网络搜索] 📖 Genius is one percent inspiration and ninety-nine percent perspiration. - Thomas 
Edison\n\n💡 中译：天才是百分之一的灵感和百分之九十九的汗水。', '活动建议': 
'建议去户外活动，享受阳光和温暖的天气。'}

[Step 4: Duration 0.88 seconds| Input tokens: 9,992 | Output tokens: 298]

KeyboardInterrupt: 

: 